In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition import LogExpectedImprovement
from botorch.optim import optimize_acqf
from gpytorch.mlls import ExactMarginalLogLikelihood



# === 1. Define Flexible Neural Network ===
class FlexibleNN(nn.Module):
    def __init__(self, input_dim, hidden_layers, output_dim, dropout_p=0.5):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_layers:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=dropout_p))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# === 2. Train & Select Best Model ===
def train_and_select_best_model(X_train_scaled, y_train_scaled, y_scaler,
                                 hidden_layers, dropout_p=0.5, lr=1e-3, weight_decay=1e-5,
                                 n_epochs=1000, n_trials=10):
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train_scaled, y_train_scaled, test_size=0.2, random_state=42)

    best_model = None
    best_val_mse = float('inf')
    best_state_dict = None

    for trial in range(n_trials):
        model = FlexibleNN(input_dim=4, hidden_layers=hidden_layers, output_dim=9, dropout_p=dropout_p)
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = nn.MSELoss()

        X_tensor = torch.tensor(X_train_split, dtype=torch.float32)
        y_tensor = torch.tensor(y_train_split, dtype=torch.float32)

        for epoch in range(n_epochs):
            model.train()
            optimizer.zero_grad()
            output = model(X_tensor)
            loss = criterion(output, y_tensor)
            loss.backward()
            optimizer.step()

        # Validation performance
        model.eval()
        with torch.no_grad():
            X_val_tensor = torch.tensor(X_val_split, dtype=torch.float32)
            y_val_pred_scaled = model(X_val_tensor).numpy()
            y_val_pred = y_scaler.inverse_transform(y_val_pred_scaled)
            y_val_real = y_scaler.inverse_transform(y_val_split)
            val_mse = mean_squared_error(y_val_real, y_val_pred)

        print(f"[Trial {trial+1}] Val MSE: {val_mse:.4f}")

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_state_dict = model.state_dict()

    best_model = FlexibleNN(4, hidden_layers, 9, dropout_p)
    best_model.load_state_dict(best_state_dict)
    print(f"\n✅ Best model selected with Val MSE: {best_val_mse:.4f}")
    return best_model


def evaluate_model(model, X_scaled, y_scaled, y_scaler, y_true=None, dataset_name="Set"):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
        y_pred_scaled = model(X_tensor).numpy()
        y_pred = y_scaler.inverse_transform(y_pred_scaled)
        y_true = y_scaler.inverse_transform(y_scaled) if y_true is None else y_true

        mse = mean_squared_error(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)

        print(f"\n--- {dataset_name} Performance ---")
        print(f"MSE: {mse:.2f}, MAE: {mae:.2f}, R²: {r2:.4f}")
        
        return y_true, y_pred

def inverse_design_bo1(model, target_props, X_train, X_scaler, y_scaler, n_init=10, n_iter=25):
    model.eval()
    input_dim = X_train.shape[1]

    # === Target property scaled ===
    target_scaled = torch.tensor(y_scaler.transform([target_props]), dtype=torch.float64)

    # === Create MinMaxScaler on standard-scaled input ===
    X_train_std = X_scaler.transform(X_train)
    minmax_scaler = MinMaxScaler()
    X_train_minmax = minmax_scaler.fit_transform(X_train_std)

    # === Initial samples in [0, 1] ===
    X_init_minmax = torch.tensor(np.random.uniform(0, 1, size=(n_init, input_dim)), dtype=torch.float64)

    def evaluate_model(x_minmax_tensor):
        x_std_np = minmax_scaler.inverse_transform(x_minmax_tensor.cpu().numpy())
        x_tensor_std = torch.tensor(x_std_np, dtype=torch.float32)  # model expects float32
        with torch.no_grad():
            y_pred = model(x_tensor_std).detach()
        return y_pred.to(dtype=torch.float64)

    # === Evaluate initial samples ===
    Y_init_model = evaluate_model(X_init_minmax)
    Y_obj = torch.mean((Y_init_model - target_scaled) ** 2, dim=1, keepdim=True)

    X_train_bo = X_init_minmax.clone()
    Y_train_bo = Y_obj.clone()

    bounds_bo = torch.tensor([[0.0] * input_dim, [1.0] * input_dim], dtype=torch.float64)

    for i in range(n_iter):
        # 1. Fit GP model
        gp = SingleTaskGP(X_train_bo, Y_train_bo)
        mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
        fit_gpytorch_mll(mll)

        # 2. Acquisition function
        acq_func = LogExpectedImprovement(gp, best_f=Y_train_bo.min().item())

        # 3. Optimize acquisition function
        candidate, _ = optimize_acqf(
            acq_function=acq_func,
            bounds=bounds_bo,
            q=1,
            num_restarts=10,
            raw_samples=100,
        )

        # 4. Evaluate new point
        y_new_model = evaluate_model(candidate)
        y_obj = torch.mean((y_new_model - target_scaled) ** 2, dim=1, keepdim=True)

        # 5. Update dataset
        X_train_bo = torch.cat([X_train_bo, candidate], dim=0)
        Y_train_bo = torch.cat([Y_train_bo, y_obj], dim=0)

        print(f"Iter {i+1:02d}: MSE = {y_obj.item():.4f}")

    # === Final result ===
    best_idx = torch.argmin(Y_train_bo)
    best_input_minmax = X_train_bo[best_idx].numpy()
    best_input_std = minmax_scaler.inverse_transform([best_input_minmax])
    best_input_real = X_scaler.inverse_transform(best_input_std)[0]

    print("\n✅ Bayesian optimization complete.")
    print("Optimized input (real scale):", best_input_real)
    return best_input_real


def inverse_design_bo2(model, target_props, X_train, X_scaler, y_scaler, n_init=10, n_iter=25, num_restarts=10):
    model.eval()
    input_dim = X_train.shape[1]

    # Scale target properties
    target_scaled = torch.tensor(y_scaler.transform([target_props]), dtype=torch.float64)

    # Scale X_train to [0, 1] for BO
    X_train_std = X_scaler.transform(X_train)
    minmax_scaler = MinMaxScaler()
    X_train_minmax = minmax_scaler.fit_transform(X_train_std)

    # Initial samples
    X_init = torch.tensor(np.random.uniform(0, 1, size=(n_init, input_dim)), dtype=torch.float64)
    with torch.no_grad():
        y_init = model(torch.tensor(
            X_scaler.inverse_transform(minmax_scaler.inverse_transform(X_init)),
            dtype=torch.float32
        )).double()

    # Train GP model
    gp = SingleTaskGP(X_init, y_init)
    mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
    fit_gpytorch_mll(mll)

    # Define acquisition function
    acq_func = LogExpectedImprovement(gp, best_f=target_scaled.mean(), maximize=False)

    # Optimize acquisition
    candidates, _ = optimize_acqf(
        acq_function=acq_func,
        bounds=torch.tensor([[0.0] * input_dim, [1.0] * input_dim], dtype=torch.float64),
        q=num_restarts,
        num_restarts=num_restarts,
        raw_samples=256,
    )

    # Map candidates back to real input scale
    candidates_std = minmax_scaler.inverse_transform(candidates.numpy())
    candidates_real = X_scaler.inverse_transform(candidates_std)

    return candidates_real



def inverse_design_bo3(model, target_props, X_train, X_scaler, y_scaler, n_init=10, n_iter=25, n_suggestions=10):

    model.eval()
    input_dim = X_train.shape[1]

    # Scale target
    target_scaled = torch.tensor(y_scaler.transform([target_props]), dtype=torch.float32)

    # Normalize X_train in [0, 1]
    X_train_std = X_scaler.transform(X_train)
    minmax_scaler = MinMaxScaler()
    X_train_minmax = minmax_scaler.fit_transform(X_train_std)

    # Generate initial samples in [0, 1]
    X_init = torch.tensor(np.random.uniform(0, 1, size=(n_init, input_dim)), dtype=torch.float32)
    
    def evaluate_points(X_norm):
        X_std = minmax_scaler.inverse_transform(X_norm)
        X_real = X_scaler.inverse_transform(X_std)
        X_tensor = torch.tensor(X_real, dtype=torch.float32)
        with torch.no_grad():
            y_pred_scaled = model(X_tensor).numpy()
        y_pred = y_scaler.inverse_transform(y_pred_scaled)
        return y_pred

    def compute_objective(y_pred):
        errors = (target_props - y_pred) ** 2
        normalized = errors / (np.array(target_props) **2)
        return np.mean(normalized, axis=1)  # return 1D array

    # Create initial dataset
    Y_init = compute_objective(evaluate_points(X_init.numpy()))
    Y_init = torch.tensor(Y_init, dtype=torch.float32).unsqueeze(-1)  # shape (n, 1)

    X = X_init.clone()
    Y = Y_init.clone()

    for i in range(n_iter):
        gp = SingleTaskGP(X, Y)
        mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
        fit_gpytorch_mll(mll)

        acquisition = LogExpectedImprovement(gp, best_f=Y.min())
        candidate, _ = optimize_acqf(
            acquisition,
            bounds=torch.stack([torch.zeros(input_dim), torch.ones(input_dim)]),
            q=1,
            num_restarts=5,
            raw_samples=100,
        )

        y_new = compute_objective(evaluate_points(candidate.detach().numpy()))
        X = torch.cat([X, candidate], dim=0)
        Y = torch.cat([Y, torch.tensor(y_new, dtype=torch.float32).unsqueeze(-1)], dim=0)

    # Select best n_suggestions points
    best_indices = torch.topk(-Y.squeeze(), n_suggestions).indices
    best_X = X[best_indices]

    # Convert from normalized → standard scaled → real scale
    best_X_std = minmax_scaler.inverse_transform(best_X.detach().numpy())
    best_X_real = X_scaler.inverse_transform(best_X_std)
    
    print("✅ Bayesian optimization complete.")
    return best_X_real


In [ ]:
import os
import time
import shutil
import subprocess
from mdsetup import MDSetup
import pandas as pd
import numpy as np

def get_job_count():
    result = subprocess.run(
        "qstat -u $USER | grep *pcff_lj_p* | wc -l",
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        universal_newlines=True
    )
    return int(result.stdout.strip())


def equil_deform_job_count():
    result = subprocess.run(
        "qstat -u $USER | grep LJ_set* | wc -l",
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        universal_newlines=True
    )
    return int(result.stdout.strip())

def SE_job_count():
    result = subprocess.run(
        "qstat -u $USER | grep SE* | wc -l",
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        universal_newlines=True
    )
    return int(result.stdout.strip())




def MD_sim(suggested_params):
    # 1. Data generation
    BASE_DIR = "input_tob_all/Suresh_data_create"
    TEMPLATE_FRC = os.path.join(BASE_DIR, "pcff_template.frc")
    TOB_DIRS = ["Tob11_car_mdf", "Tob11H_car_mdf", "Tob14_car_mdf"]

    lj_dir = [d for d in os.listdir(f"{BASE_DIR}/..") if os.path.isdir(os.path.join(BASE_DIR, "..", d)) and d.startswith("LJ_set_")]
    count1 = len(lj_dir)

    for i, (sc_r, sc_eps, oc_r, oc_eps) in enumerate(zip(*suggested_params.T)):
        i += count1
        param_dir = os.path.join(BASE_DIR, f"../LJ_set_{i+1}")
        os.makedirs(param_dir, exist_ok=True)

        # Modify FRC file
        with open(TEMPLATE_FRC, "r") as f:
            frc_content = f.read()
        
        # Replace placeholders
        frc_content = frc_content.replace("{sc_r}", str(sc_r))
        frc_content = frc_content.replace("{sc_eps}", str(sc_eps))
        frc_content = frc_content.replace("{oc_r}", str(oc_r))
        frc_content = frc_content.replace("{oc_eps}", str(oc_eps))
        
        frc_path = os.path.join(param_dir, f"modified_LJ_{i+1}.frc")
        with open(frc_path, "w") as f:
            f.write(frc_content)

        # Run msi2lmp
        for tob_dir in TOB_DIRS:
            dir_path = os.path.join(BASE_DIR, tob_dir)
            output_folder = os.path.join(param_dir, tob_dir.split("_")[0])
            os.makedirs(output_folder, exist_ok=True)

            prefixes = sorted(set("_".join(f.split(".")[:-1]) for f in os.listdir(dir_path) if f.endswith((".car", ".mdf"))))
            for prefix in prefixes:
                output_prefix = os.path.join(dir_path, prefix)
                cmd = ["msi2lmp", output_prefix, "-class", "2", "-frc", os.path.join(param_dir, f"modified_LJ_{i+1}.frc"), "-ignore"]
                print(f"Running: {' '.join(cmd)} with LJ_set_{i+1}")
                result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)

                if result.returncode == 0:
                    print(f"✅ Successfully generated {prefix}.data for LJ_set_{i+1}")
                    try:
                        shutil.move(f"{output_prefix}.data", output_folder)
                    except Exception as e:
                        print(f"⚠️ Unexpected error: {e}")
                else:
                    print(f"❌ Error in {prefix}: {result.stderr}")

    count2 = count1 + len(suggested_params)

    # 2. MDSetup
    os.chdir('')
    lammps_setup = MDSetup(
        system_setup="input_tob_all/setup_mechanical_pcff.yaml",
        simulation_default="input_tob_all/defaults.yaml",
        simulation_ensemble="input_tob_all/ensemble.yaml",
        simulation_sampling="input_tob_all/sampling_mechanical.yaml",
        submission_command="qsub",
    )

    LJ_SETS = [f"LJ_set_{i+1}" for i in range(count1, count2)]

    # 2.1. Equilibration Setup
    TOB_STRUCTURES = {
        "Tob11": "tob11_221.data",
        "Tob11H": "tob11H_221.data",
        "Tob14": "tob14_221.data",
    }

    for lj_set in LJ_SETS:
        for tob_structure, data_file in TOB_STRUCTURES.items():

            lammps_setup.prepare_simulation(
                folder_name=f"{tob_structure}/{lj_set}/equilibration",
                ensembles=["em", "npt"],
                simulation_times=[0.1, 1.0],
                initial_systems=[f"input_tob_all/{lj_set}/{tob_structure}/{data_file}"],
                input_kwargs={},
                copies=0,
                off_set=0,
            )
            lammps_setup.submit_simulation()
            print(f"✅ Submitted job for {tob_structure} with {lj_set}")

    # 2.2.(i). Surface Energy - Bulk
    SE_STRUCTURES_BULK = {
        "Tob11": "Tob11_004_Bulk_cV05_r00.data",
        "Tob11H": "Tob11_Hamid_004_002_Bulk_cV02_r00.data",
        "Tob14": "Tob14_001_004_Bulk_cV07_r01.data",
    }

    for lj_set in LJ_SETS:
        for tob_structure, data_file in SE_STRUCTURES_BULK.items():
            lammps_setup.prepare_simulation(
                folder_name=f"{tob_structure}/{lj_set}/SE/bulk",
                ensembles=["nvt"],
                simulation_times=[1.0],
                initial_systems=[f"input_tob_all/{lj_set}/{tob_structure}/{data_file}"],
                input_kwargs={},
                copies=0,
                off_set=0,
            )
            lammps_setup.submit_simulation()
            print(f"✅ Submitted job for {tob_structure} with {lj_set}")

    # 2.2.(ii). Surface Energy - Cleaved
    SE_STRUCTURES_VACUUM = {
        "Tob11": "Tob11_004_vacuum_cV05_r01.data",
        "Tob11H": "Tob11_Hamid_004_Cleaved_cV02_r00.data",
        "Tob14": "Tob14_004_vacuum_cV07_r01.data",
    }

    for lj_set in LJ_SETS:
        for tob_structure, data_file in SE_STRUCTURES_VACUUM.items():
            lammps_setup.prepare_simulation(
                folder_name=f"{tob_structure}/{lj_set}/SE/vacuum",
                ensembles=["nvt"],
                simulation_times=[1.0],
                initial_systems=[f"input_tob_all/{lj_set}/{tob_structure}/{data_file}"],
                input_kwargs={},
                copies=0,
                off_set=0,
            )
            lammps_setup.submit_simulation()
            print(f"✅ Submitted job for {tob_structure} with {lj_set}")

    # 2.3. Deformation Jobs
    while True:
        # if get_job_count() <= len(suggested_params) * len(TOB_STRUCTURES) * 2:
        if equil_deform_job_count() <= 0:
            print("Submitting deformation jobs...")
            deformation_directions = ["xx", "yy", "zz", "xy", "xz", "yz", "undeformed"]
            deformation_rates = [-0.02, -0.01, 0.00, 0.01, 0.02]

            for lj_set in LJ_SETS:
                for tob_structure, data_file in TOB_STRUCTURES.items():
                    initial_systems = [f"output_tob_all/pcff_lj_params/{tob_structure}/{lj_set}/equilibration/temp_298.1_pres_1.0/copy_0/01_npt/npt.data"]
                    job_files = [[] for _ in lammps_setup.system_setup["temperature"]]

                    for direction in deformation_directions:
                        for rate in deformation_rates:
                            if (rate == 0.0 and direction != "undeformed") or (rate != 0.0 and direction == "undeformed"):
                                continue
                            folder = f"{tob_structure}/{lj_set}/deformation/{direction}/{rate}"
                            input_kwargs = {"deformation": {"direction": direction, "rate": rate}}

                            lammps_setup.prepare_simulation(
                                folder_name=folder,
                                ensembles=["nvt"],
                                simulation_times=[1.0],
                                initial_systems=initial_systems,
                                input_kwargs=input_kwargs,
                                copies=0,
                                off_set=0,
                            )
                            for j, files in enumerate(lammps_setup.job_files):
                                job_files[j].extend(files)

                    lammps_setup.job_files = job_files
                    lammps_setup.submit_simulation(individual_sub=False)
                    print(f"✅ Submitted deformation jobs for {tob_structure} with {lj_set}")
            break
        else:
            # print(f"{get_job_count()} jobs still running. Waiting (for deformation)...")
            print(f"{equil_deform_job_count()} equilibrium jobs still running. Waiting (for deformation)...")
            print(f"total {get_job_count()} jobs are running")
            time.sleep(30)
    
    print("All jobs submitted successfully.")
    

    #3. Analysis
    while True:
        # if get_job_count() <= len(suggested_params) * len(TOB_STRUCTURES) * 2:
        if equil_deform_job_count() <= 0:
            
            TOB_STRUCTURES = ["Tob11", "Tob11H", "Tob14"]
            # 1. Density Analysis
            den_results = {tob: [] for tob in TOB_STRUCTURES}

            for lj_set in LJ_SETS:
                for tob_structure in TOB_STRUCTURES:
                    analysis_folder = f"{tob_structure}/{lj_set}/equilibration"    
                    extracted_values = lammps_setup.analysis_extract_properties(
                        analysis_folder=analysis_folder,
                        ensemble='01_npt',
                        extracted_properties=['density'],
                        output_suffix='density',
                        time_fraction=0.2,
                    )
                    average_values = extracted_values.get('01_npt', {}).get("data", {}).get("average", {})
                    mean_value = average_values.get('density', {}).get("mean", None)
                    den_results[tob_structure].append(mean_value)
    
            # 2. Surface Energy Analysis
            NA = 6.022e23
            CONVERSION = 4184  # kcal/mol to J/mol

            # Box dimensions (structure-dependent)
            box_coords = {
                "Tob11":    [0.527972175, 23.057572175, -0.431033519, 21.723966481],
                "Tob11H":   [0.065633394, 22.383633394, -0.317852898, 29.242147102],
                "Tob14":    [-0.658063909, 21.871536091, -0.289451837, 21.985548163],
            }

            se_results = {struct: [] for struct in TOB_STRUCTURES}
            for struct in TOB_STRUCTURES:
                for lj in LJ_SETS:
                    try:
                        base = f"{struct}/{lj}/SE"
                        
                        state = 'bulk'
                        folder = f"{base}/{state}"
                        extracted_values = lammps_setup.analysis_extract_properties(
                            analysis_folder=folder,
                            ensemble="00_nvt",
                            extracted_properties=["potential energy"],
                            output_suffix="energy",
                            time_fraction=0.4,
                        )
                        bulk_energy = extracted_values.get("00_nvt", {}).get("data", {}).get("average", {}).get("potential energy", {})

                        state = 'vacuum'
                        folder = f"{base}/{state}"
                        extracted_values = lammps_setup.analysis_extract_properties(
                            analysis_folder=folder,
                            ensemble="00_nvt",
                            extracted_properties=["potential energy"],
                            output_suffix="energy",
                            time_fraction=0.4,
                        )
                        vac_energy = extracted_values.get("00_nvt", {}).get("data", {}).get("average", {}).get("potential energy", {})
                        
                        E_bulk, SD_bulk = bulk_energy["mean"], bulk_energy["std"]
                        E_vac, SD_vac = vac_energy["mean"], vac_energy["std"]

                        # Compute surface area
                        xlo, xhi, ylo, yhi = box_coords[struct]
                        A = abs(xhi - xlo) * abs(yhi - ylo) * 1e-20  # m²

                        # Compute SE and std dev
                        deltaE = 1000 * (E_vac - E_bulk) * CONVERSION / (2 * A * NA)  # mJ/m²
                        s_dev = 1000 * (SD_bulk + SD_vac) * CONVERSION / (2 * A * NA)

                        se_results[struct].append((deltaE, s_dev))
                        print(f"✅ Analyzed {struct} with {lj}: SE = {deltaE:.2f} ± {s_dev:.2f} mJ/m²")
                    
                    except Exception as e:
                        print(f"❌ Failed to analyze {struct} with {lj}: {e}")
                        se_results[struct].append((None, None))
                        continue
            
            # 3. Mechanical Properties Analysis            
            bm_results = {tob: [] for tob in TOB_STRUCTURES}
            for lj_set in LJ_SETS:
                for tob_structure in TOB_STRUCTURES:
                    try:
                        # Define the deformation analysis folder
                        analysis_folder = f"{tob_structure}/{lj_set}/deformation"
                        # Run analysis
                        BM, Cij = lammps_setup.analysis_mechanical_proerties(
                            analysis_folder=analysis_folder,
                            ensemble="00_nvt",
                            deformation_rates=[-0.02, -0.01, 0.00, 0.01, 0.02],
                            method="VRH",
                            time_fraction=0.4,
                            visualize_stress_strain=False,
                        )

                        print(f"✅ Analyzed {tob_structure} with {lj_set}: BM = {BM:.2f} GPa")
                        bm_results[tob_structure].append(BM)

                    except Exception as e:
                        print(f"❌ Failed to analyze {tob_structure} with {lj_set}: {e}")
                        bm_results[tob_structure].append(None)
                        continue

            sc_r, sc_eps, oc_r, oc_eps =  suggested_params[:, 0], suggested_params[:, 1], suggested_params[:, 2], suggested_params[:, 3]
            
            
            df = pd.DataFrame({
                'sc_r': sc_r,
                'sc_eps': sc_eps,
                'oc_r': oc_r,
                'oc_eps': oc_eps,
                
                'D_11': np.array(den_results['Tob11']),
                'D_11H': np.array(den_results['Tob11H']),
                'D_14': np.array(den_results['Tob14']),
                
                'SE_T11': np.array(se_results['Tob11'])[:,0],
                'SE_T11H': np.array(se_results['Tob11H'])[:,0],
                'SE_T14': np.array(se_results['Tob14'])[:,0],
                
                'BM_T11': np.array(bm_results['Tob11']),
                'BM_T11H': np.array(bm_results['Tob11H']),
                'BM_T14': np.array(bm_results['Tob14'])
                
            })
            # save the df as a csv file with the name "data_time.csv"
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            df.to_csv(f"output_tob_all/pcff_lj_params/data_{timestamp}.csv", index=False)
            print(f"✅ All jobs completed and data saved as data_{timestamp}.csv")
            return df
        
        else:
            # print(f"{get_job_count()} jobs still running. Waiting...(for Analysis)")
            print(f"{equil_deform_job_count()} equilibrium and deformation jobs still running. Waiting (for Analysis)...")
            print(f"total {get_job_count()} jobs are running")
            time.sleep(30)



In [ ]:
# my_range = range(135, 145)
my_range = range(123, 135)
LJ_SETS = [f"LJ_set_{i+1}" for i in my_range]

# 2. MDSetup
os.chdir('')
lammps_setup = MDSetup(
    system_setup="input_tob_all/setup_mechanical_pcff.yaml",
    simulation_default="input_tob_all/defaults.yaml",
    simulation_ensemble="input_tob_all/ensemble.yaml",
    simulation_sampling="input_tob_all/sampling_mechanical.yaml",
    submission_command="qsub",
)


TOB_STRUCTURES = ["Tob11", "Tob11H", "Tob14"]
# 1. Density Analysis
den_results = {tob: [] for tob in TOB_STRUCTURES}

for lj_set in LJ_SETS:
    for tob_structure in TOB_STRUCTURES:
        analysis_folder = f"{tob_structure}/{lj_set}/equilibration"    
        extracted_values = lammps_setup.analysis_extract_properties(
            analysis_folder=analysis_folder,
            ensemble='01_npt',
            extracted_properties=['density'],
            output_suffix='density',
            time_fraction=0.2,
        )
        average_values = extracted_values.get('01_npt', {}).get("data", {}).get("average", {})
        mean_value = average_values.get('density', {}).get("mean", None)
        den_results[tob_structure].append(mean_value)

# 2. Surface Energy Analysis
NA = 6.022e23
CONVERSION = 4184  # kcal/mol to J/mol

# Box dimensions (structure-dependent)
box_coords = {
    "Tob11":    [0.527972175, 23.057572175, -0.431033519, 21.723966481],
    "Tob11H":   [0.065633394, 22.383633394, -0.317852898, 29.242147102],
    "Tob14":    [-0.658063909, 21.871536091, -0.289451837, 21.985548163],
}

se_results = {struct: [] for struct in TOB_STRUCTURES}
for struct in TOB_STRUCTURES:
    for lj in LJ_SETS:
        try:
            base = f"{struct}/{lj}/SE"
            
            state = 'bulk'
            folder = f"{base}/{state}"
            extracted_values = lammps_setup.analysis_extract_properties(
                analysis_folder=folder,
                ensemble="00_nvt",
                extracted_properties=["potential energy"],
                output_suffix="energy",
                time_fraction=0.4,
            )
            bulk_energy = extracted_values.get("00_nvt", {}).get("data", {}).get("average", {}).get("potential energy", {})

            state = 'vacuum'
            folder = f"{base}/{state}"
            extracted_values = lammps_setup.analysis_extract_properties(
                analysis_folder=folder,
                ensemble="00_nvt",
                extracted_properties=["potential energy"],
                output_suffix="energy",
                time_fraction=0.4,
            )
            vac_energy = extracted_values.get("00_nvt", {}).get("data", {}).get("average", {}).get("potential energy", {})
            
            E_bulk, SD_bulk = bulk_energy["mean"], bulk_energy["std"]
            E_vac, SD_vac = vac_energy["mean"], vac_energy["std"]

            # Compute surface area
            xlo, xhi, ylo, yhi = box_coords[struct]
            A = abs(xhi - xlo) * abs(yhi - ylo) * 1e-20  # m²

            # Compute SE and std dev
            deltaE = 1000 * (E_vac - E_bulk) * CONVERSION / (2 * A * NA)  # mJ/m²
            s_dev = 1000 * (SD_bulk + SD_vac) * CONVERSION / (2 * A * NA)

            se_results[struct].append((deltaE, s_dev))
            print(f"✅ Analyzed {struct} with {lj}: SE = {deltaE:.2f} ± {s_dev:.2f} mJ/m²")
        
        except Exception as e:
            print(f"❌ Failed to analyze {struct} with {lj}: {e}")
            se_results[struct].append((None, None))
            continue

# 3. Mechanical Properties Analysis            
bm_results = {tob: [] for tob in TOB_STRUCTURES}
for lj_set in LJ_SETS:
    for tob_structure in TOB_STRUCTURES:
        try:
            # Define the deformation analysis folder
            analysis_folder = f"{tob_structure}/{lj_set}/deformation"
            # Run analysis
            BM, Cij = lammps_setup.analysis_mechanical_proerties(
                analysis_folder=analysis_folder,
                ensemble="00_nvt",
                deformation_rates=[-0.02, -0.01, 0.00, 0.01, 0.02],
                method="VRH",
                time_fraction=0.4,
                visualize_stress_strain=False,
            )

            print(f"✅ Analyzed {tob_structure} with {lj_set}: BM = {BM:.2f} GPa")
            bm_results[tob_structure].append(BM)

        except Exception as e:
            print(f"❌ Failed to analyze {tob_structure} with {lj_set}: {e}")
            bm_results[tob_structure].append(None)
            continue

import os

BASE_DIR = "input_tob_all/Suresh_data_create"

sc_r_values_n = []
sc_eps_values_n = []
oc_r_values_n = []
oc_eps_values_n = []

for i in my_range:
    param_dir = os.path.join(BASE_DIR, f"../LJ_set_{i+1}")
    frc_path = os.path.join(param_dir, f"modified_LJ_{i+1}.frc")

    with open(frc_path, "r") as file:
        lines = file.readlines()

    sc_r, sc_eps = [float(i) for i in lines[3954].strip().split()[-2:]]
    oc_r, oc_eps = [float(i) for i in lines[3958].strip().split()[-2:]]

    sc_r_values_n.append(sc_r)
    sc_eps_values_n.append(sc_eps)
    oc_r_values_n.append(oc_r)
    oc_eps_values_n.append(oc_eps)



sc_r, sc_eps, oc_r, oc_eps =  np.array(sc_r_values_n), np.array(sc_eps_values_n), np.array(oc_r_values_n), np.array(oc_eps_values_n)


df = pd.DataFrame({
    'sc_r': sc_r,
    'sc_eps': sc_eps,
    'oc_r': oc_r,
    'oc_eps': oc_eps,
    
    'D_11': np.array(den_results['Tob11']),
    'D_11H': np.array(den_results['Tob11H']),
    'D_14': np.array(den_results['Tob14']),
    
    'SE_T11': np.array(se_results['Tob11'])[:,0],
    'SE_T11H': np.array(se_results['Tob11H'])[:,0],
    'SE_T14': np.array(se_results['Tob14'])[:,0],
    
    'BM_T11': np.array(bm_results['Tob11']),
    'BM_T11H': np.array(bm_results['Tob11H']),
    'BM_T14': np.array(bm_results['Tob14'])
    
})
# save the df as a csv file with the name "data_time.csv"
timestamp = time.strftime("%Y_%m_%d_%H_%M_%S")
df.to_csv(f"output_tob_all/pcff_lj_params/data_{timestamp}.csv", index=False)
print(f"✅ All jobs completed and data saved as data_{timestamp}.csv")



In [ ]:
df_new = df.copy()

In [ ]:
target_y = [2.46, 2.39, 2.23, 680, 325, 635, 71, 55.35, 47]

md_prop_pred = df_new.values[:,4:]
y_error = np.abs(100*(md_prop_pred - np.array(target_y))/np.array(target_y))

for i in range(len(y_error)):
    targeted_error = np.array([1, 1, 1, 5, 5, 5, 10, 10, 10])
    if (y_error[i] <= targeted_error).sum() == 9:
        print("All properties are within error limits")
        print(df_new.iloc[i])
        break # break Active learning loop.

    else:
        total_false = (~(y_error[i] <= targeted_error)).sum()
        print("❌ Failure! Outside acceptable error range.")
        print(f"Failed for {total_false} properties")
        print("Error percentage:", y_error[i].round(2))

# === 8. Append new data ===
df = pd.concat([df, df_new], ignore_index=True)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os 
os.chdir('ML_Pipeline')

# === 1. Load Data ===
df = pd.read_csv('data/training_data_123.csv').dropna()

X = df.values[:, :4]
y = df.values[:, 4:]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Assuming y is your (114, 9) numpy array
# Step 1: Standardize the data
scaler = StandardScaler()
y_scaled = scaler.fit_transform(y)

# Step 2: Compute PCA
pca = PCA()
pca.fit(y_scaled)

# Step 3: Extract eigenvalues (i.e., explained variance)
# eigenvalues = pca.explained_variance_
explained_variance_ratio = pca.explained_variance_ratio_

# Optional: Plot cumulative explained variance
plt.figure(figsize=(8, 5))
plt.plot(np.cumsum(explained_variance_ratio), marker='o', linestyle='--')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance vs Number of Components')
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

os.chdir('ML_Pipeline')

# === 1. Load Data ===
df = pd.read_csv('data/training_data_123.csv')

for AL_run in range(2):
    df = df.dropna()
    X = df.values[:, :4]
    y = df.values[:, 4:]

    # === 2. Split into train and test ===
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # === 3. Scale ===
    X_scaler = StandardScaler().fit(X_train)
    y_scaler = StandardScaler().fit(y_train)

    X_train_scaled = X_scaler.transform(X_train)
    y_train_scaled = y_scaler.transform(y_train)
    X_test_scaled = X_scaler.transform(X_test)
    y_test_scaled = y_scaler.transform(y_test)
    
    # === 4. Train model ===
    model = train_and_select_best_model(
        X_train_scaled, y_train_scaled, y_scaler,
        hidden_layers=[64, 32],  # customize this
        dropout_p=0.3,                # try 0.2, 0.3, 0.5
        lr=5e-4,                      # try 1e-3, 5e-4, 1e-4
        weight_decay=1e-4,               # L2 regularization
        n_epochs=1000,
        n_trials=10
    )

     # === 5. Evaluate model ===
    y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_scaler, dataset_name="Train")
    y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_scaler, dataset_name="Test")

    # === 6. Inverse Design using BO ===
    target_y = [2.46, 2.39, 2.23, 680, 325, 635, 71, 55.35, 47]
    x_opt_bo = inverse_design_bo3(
        model=model,
        target_props=target_y,
        X_train=X_train,
        X_scaler=X_scaler,
        y_scaler=y_scaler,
        n_init=10,
        n_iter=25,
        n_suggestions=10
    )
    
    df_new = MD_sim(np.atleast_2d(x_opt_bo))
    
    # ################################
    # X_bo_scaled = X_scaler.transform(x_opt_bo)
    # model.eval()
    # with torch.no_grad():
    #     X_tensor = torch.tensor(X_bo_scaled, dtype=torch.float32)
    #     y_pred_scaled = model(X_tensor).numpy()
    #     y_pred = y_scaler.inverse_transform(y_pred_scaled)
            
    # aa = df.columns.to_list()
    # df_1 = pd.DataFrame(x_opt_bo)
    # df_2 = pd.DataFrame(y_pred)
    # df_new = pd.concat([df_1.T, df_2.T], ignore_index=True).T
    # df_new = df_new.rename(columns={0: aa[0], 1: aa[1], 2: aa[2], 3: aa[3], 4: aa[4], 5: aa[5], 6: aa[6], 7: aa[7], 8: aa[8], 9: aa[9], 10: aa[10], 11:aa[11], 12:aa[12]})
    # ################################
    
    md_prop_pred = df_new.values[:,4:]
    y_error = np.abs(100*(md_prop_pred - np.array(target_y))/np.array(target_y))

    for i in range(len(y_error)):
        targeted_error = np.array([1, 1, 1, 5, 5, 5, 10, 10, 10])
        if (y_error[i] <= targeted_error).sum() == 9:
            print("All properties are within error limits")
            print(df_new.iloc[i])
            break # break Active learning loop.

        else:
            total_false = (~(y_error[i] <= targeted_error)).sum()
            print("❌ Failure! Outside acceptable error range.")
            print(f"Failed for {total_false} properties")
            print("Error percentage:", y_error[i].round(2))
    
    # === 8. Append new data ===
    df = pd.concat([df, df_new], ignore_index=True)
